In [0]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [0]:
catalog_name = "ecommerce"

In [0]:
df = spark.table(f'{catalog_name}.bronze.brz_order_items')
display(df)

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,1,2000000028279,1,GBP,11,10%,2,web,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,2,2000000377445,1,GBP,5,4%,1,web,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 09:44:32,CUST000000239553,643612,1,2000000417639,4,INR,4871,8%,3245,web,FEST20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,1,2000000422664,Two,AED,74,0%,18,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,2,2000000238159,1,AED,1012,11%,109,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,3,2000000094205,Two,AED,229,15%,20,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,4,2000000293790,Two,AED,2222,0%,801,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 21:09:10,CUST000000117094,643614,1,2000000271415,3,INR,252,10%,82,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 13:03:57,CUST000000016826,643615,1,2000000050980,1,GBP,71,5%,9,web,SAVE50,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z
2025-08-01,2025-08-01 17:54:51,CUST000000252796,643616,1,2000000463254,1,USD,$6,9%,1,app,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z


In [0]:
df.printSchema()

root
 |-- dt: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- item_seq: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price_currency: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



In [0]:
df.describe().show()

+-------+----------+-------------------+----------------+-----------------+------------------+--------------------+------------------+-------------------+-----------------+------------+-----------------+-------+-----------+--------------------+
|summary|        dt|           order_ts|     customer_id|         order_id|          item_seq|          product_id|          quantity|unit_price_currency|       unit_price|discount_pct|       tax_amount|channel|coupon_code|           file_name|
+-------+----------+-------------------+----------------+-----------------+------------------+--------------------+------------------+-------------------+-----------------+------------+-----------------+-------+-----------+--------------------+
|  count|    183378|             183378|          183378|           183378|            183378|              183378|            183378|             183378|           183378|      183378|           183378| 183378|      63830|              183378|
|   mean|      NULL|

In [0]:
# Count duplicate (non-null) values for each column
for col_name in df.columns:
    dup_count = (
        df.filter(F.col(col_name).isNotNull())
          .groupBy(col_name)
          .agg(F.count(F.lit(1)).alias("cnt"))
          .filter(F.col("cnt") > 1)
          .count()
    )
    print(f"{col_name}: {dup_count} duplicate values")


dt: 92 duplicate values
order_ts: 47155 duplicate values
customer_id: 47678 duplicate values
order_id: 47083 duplicate values
item_seq: 5 duplicate values
product_id: 44001 duplicate values
quantity: 4 duplicate values
unit_price_currency: 7 duplicate values
unit_price: 13326 duplicate values
discount_pct: 41 duplicate values
tax_amount: 6358 duplicate values
channel: 2 duplicate values
coupon_code: 4 duplicate values
file_name: 92 duplicate values
ingest_timestamp: 1 duplicate values


### Transforming the columns


In [0]:
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

In [0]:
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

In [0]:
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

In [0]:
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

In [0]:
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

### Datatype Conversion

In [0]:
df = df.withColumn(
    "dt",
    F.to_date("dt", "yyyy-MM-dd")     
)

In [0]:
df = df.withColumn(
    "order_ts",
    F.coalesce(
        F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),  # matches 2025-08-01 22:53:52
        F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")      # fallback for 01-08-2025 22:53
    )
)

In [0]:
df = df.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)


In [0]:
df = df.withColumn(
    "tax_amount",
    F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double")
)


In [0]:
df = df.withColumn(
    "processed_time", F.current_timestamp()
)

In [0]:
display(df.limit(5))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:23:10.859Z
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:23:10.859Z
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:23:10.859Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:23:10.859Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:23:10.859Z


In [0]:
df.printSchema()

root
 |-- dt: date (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- item_seq: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_currency: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- processed_time: timestamp (nullable = false)



In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")